In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Load Dataset
df = pd.read_csv("CAR DETAILS FROM CAR DEKHO.csv")

# Display first 5 rows
print("First 5 Rows:")
display(df.head())

# Dataset Shape
print(f"\nDataset Shape: {df.shape}")

# Column Names
print("\nColumns:")
print(df.columns.tolist())

# Dataset Information
print("\nDataset Information:")
df.info()

# Statistical Summary
print("\nStatistical Summary:")
display(df.describe())


ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Check missing values
print("Missing Values:")
print(df.isnull().sum())

# Check duplicate rows
print("\nDuplicate Rows:", df.duplicated().sum())

# Remove duplicate rows
df.drop_duplicates(inplace=True)

# Fill missing values
for col in df.select_dtypes(include=["object", "string"]):
    df[col] = df[col].fillna(df[col].mode()[0])

for col in df.select_dtypes(include=["int64", "float64"]):
    df[col] = df[col].fillna(df[col].median())

# Check if missing values are handled
print("\nMissing Values After Cleaning:")
print(df.isnull().sum())

# Display cleaned dataset
display(df.head())


In [ ]:
# Create Car Age Feature
from datetime import datetime
current_year = datetime.now().year
df["Car_Age"] = current_year - df["year"]

# Extract Brand Name from Car Name
df["Brand"] = df["name"].str.split().str[0]

# Display Updated Dataset
display(df.head())


In [ ]:
# Selling Price Distribution
plt.figure(figsize=(8,5))

plt.hist(df["selling_price"], bins=30)

plt.title("Selling Price Distribution")
plt.xlabel("Selling Price")
plt.ylabel("Count")
plt.grid(alpha=0.3)
plt.show()

# Fuel Type vs Selling Price
plt.figure(figsize=(8,5))
sns.boxplot(x="fuel", y="selling_price", data=df)
plt.title("Fuel Type vs Selling Price")
plt.xticks(rotation=45)
plt.show()

# Transmission vs Selling Price
plt.figure(figsize=(8,5))
sns.boxplot(x="transmission", y="selling_price", data=df)
plt.title("Transmission vs Selling Price")
plt.show()

# Car Age vs Selling Price
plt.figure(figsize=(8,5))
sns.scatterplot(x="Car_Age", y="selling_price", data=df)
plt.title("Car Age vs Selling Price")
plt.show()

# Correlation Heatmap
plt.figure(figsize=(8,6))
sns.heatmap(df.select_dtypes(include=np.number).corr(),
            annot=True,
            cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()


In [ ]:
# Separate Features and Target
X = df.drop(["selling_price", "name"], axis=1)
y = df["selling_price"]

# Identify Categorical and Numerical Columns
categorical_cols = X.select_dtypes(include=["object","string"]).columns
numerical_cols = X.select_dtypes(exclude=["object","string"]).columns

# One-Hot Encoding
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ],
    remainder="passthrough"
)

print("Categorical Columns:", list(categorical_cols))
print("Numerical Columns:", list(numerical_cols))


In [ ]:
# Split Dataset into Training and Testing Sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)


In [ ]:
# Linear Regression Model
lr_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

# Train Model
lr_model.fit(X_train, y_train)

print("Linear Regression Model Trained Successfully!")


In [ ]:
# Random Forest Model
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

# Train Model
rf_model.fit(X_train, y_train)

print("Random Forest Model Trained Successfully!")


In [ ]:
# Predict using Linear Regression
lr_predictions = lr_model.predict(X_test)

# Evaluation Metrics
lr_mae = mean_absolute_error(y_test, lr_predictions)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_predictions))
lr_r2 = r2_score(y_test, lr_predictions)

print("Linear Regression Performance")
print("-" * 35)
print("Mean Absolute Error (MAE):", lr_mae)
print("Root Mean Squared Error (RMSE):", lr_rmse)
print("R² Score:", lr_r2)


In [ ]:
# Predict using Random Forest
rf_predictions = rf_model.predict(X_test)

# Evaluation Metrics
rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))
rf_r2 = r2_score(y_test, rf_predictions)

print("Random Forest Performance")
print("-" * 35)
print("Mean Absolute Error (MAE):", rf_mae)
print("Root Mean Squared Error (RMSE):", rf_rmse)
print("R² Score:", rf_r2)


In [ ]:
# Compare Models
comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [lr_mae, rf_mae],
    "RMSE": [lr_rmse, rf_rmse],
    "R² Score": [lr_r2, rf_r2]
})

comparison


In [ ]:
# Extract Feature Names
encoder = rf_model.named_steps["preprocessor"].named_transformers_["cat"]

feature_names = list(encoder.get_feature_names_out(categorical_cols)) + list(numerical_cols)

# Get Feature Importance
importance = rf_model.named_steps["model"].feature_importances_

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance.head(15)


In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(
    data=feature_importance.head(15),
    x="Importance",
    y="Feature"
)

plt.title("Top 15 Important Features")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()


In [ ]:
# Predict Selling Price for First Car
sample = X.iloc[[0]]

predicted_price = rf_model.predict(sample)

print("Predicted Selling Price: ₹{:,.2f}".format(predicted_price[0]))
print("Actual Selling Price: ₹{:,.2f}".format(y.iloc[0]))
